<a href="https://colab.research.google.com/github/sascha-pst/Jupyter-Notebooks/blob/main/FoodAggregator_Project_Polished.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FoodHub Order Analysis: Statistical Inference and Predictive Modeling

> A statistical and machine-learning analysis of 1,898 food delivery orders from FoodHub, a New York food aggregator app. Beyond describing demand, delivery speed, ratings and revenue, this report **evaluates associations and sampling uncertainty** (hypothesis tests, bootstrap confidence intervals, effect sizes, multiple-testing control) and **quantifies what the data can and cannot predict** (cross-validated regression and classification models), then translates the evidence into business recommendations.

## Executive Summary

| # | Finding | Evidence | Statistical / ML method | So what |
|---|---|---|---|---|
| 1 | **Demand is concentrated** | 4 cuisines (American, Japanese, Italian, Chinese) make up **83%** of orders; Shake Shack alone is **11.5%**; most orders fall on weekends | Gini coefficient, effective number of restaurants (inverse HHI), Lorenz curve, exact binomial test (§3.1) | Partnerships with a few restaurants move the needle, but create dependency risk |
| 2 | **Weekday deliveries are slower** | Weekday mean delivery **28.3 min** vs weekend **22.5 min** (~6 min longer) | Welch t-test, Mann-Whitney U, bootstrap CI, Cohen's d, prep-time comparison (§4.1); covariate-adjusted regression (§8.2) | Investigate weekday routing, distance and driver availability |
| 3 | **Customers who rate are satisfied, but most don't rate** | **39%** of orders unrated; among rated orders ~**51%** are 5-star and **none are below 3** | Chi-square and Mann-Whitney non-response diagnostics, Spearman correlations with Holm correction, power analysis (§5.2); classifier with permutation test (§8.3) | Improve feedback coverage; missing ratings do not establish satisfaction |
| 4 | **Service is mostly fast** | Only **10.5%** of orders exceed 60 min from placement to delivery | Empirical distribution; cross-validated delivery-time models and permutation importance (§8.1) | Validate service-time coverage before making a marketing promise |
| 5 | **Higher-value orders drive revenue** | **29%** of orders exceed \$20 and earn the higher 25% commission, generating **60%** of revenue | Commission simulation, bootstrap CIs on revenue shares, Kruskal-Wallis across cuisines (§6.1) | Test basket-building offers against incremental margin and conversion |

The **evidence scorecard in §9** consolidates every test and model result, with effect sizes, confidence intervals and corrected p-values, and is generated from the data each time the notebook runs.

### Analytical toolkit

| Technique | Purpose in this report | Section |
|---|---|---|
| Gini coefficient, Lorenz curve, effective number (1/HHI) | Measure how unequal restaurant and cuisine demand is | 3.1 |
| Welch t-test and Mann-Whitney U | Compare groups with and without a normality assumption | 4.1, 5.2 |
| Bootstrap confidence intervals (10,000 resamples) | Distribution-free uncertainty for means, differences and revenue shares | 4.1, 6.1, 8.2 |
| Cohen's d, Cramer's V, epsilon-squared | Effect sizes, so significance is not confused with importance | 4.1, 5.2, 6.1 |
| Chi-square test of independence | Test whether non-response depends on day or cuisine | 5.2 |
| Spearman rank correlation and Kruskal-Wallis | Association tests suited to an ordinal 3-5 rating scale | 5.2, 6.1 |
| Holm-Bonferroni correction | Control false positives when running families of tests | 4.1, 5.2, 7.1 |
| Power analysis (Fisher z) | Approximate detectable correlation under simplifying assumptions | 5.2 |
| One-sided t-test with confidence intervals | Check that promotion picks clear the 4.0 bar beyond sampling noise | 7.1 |
| Ridge, random forest, gradient boosting regression | Predict delivery time; benchmark against a naive baseline | 8.1 |
| Repeated 5-fold cross-validation inside pipelines with fold-specific preprocessing | Compare validation errors; random folds may share customers | 8.1, 8.3 |
| Permutation importance | Explore feature importance after model comparison | 8.1 |
| OLS with bootstrap CI | Adjust the weekend effect for cuisine, order size and prep time | 8.2 |
| Logistic regression and gradient boosting classifiers; permutation test | Test whether any available data predicts a 5-star rating | 8.3 |

**How to read the evidence.** Tests are two-sided at the 5% level unless stated, with Holm correction applied within each family of related tests. Bootstrap results use 10,000 resamples for the day-type comparisons, 5,000 for revenue and non-response comparisons, and 2,000 for adjusted regression with a fixed seed so every run reproduces. Models are scored on data they were not trained on. The data are observational, so findings describe associations, not proven causes.

## Business Context

FoodHub connects students and professionals with a wide range of New York restaurants. The company wants to use its order data to understand restaurant demand, improve customer experience, and strengthen operations in a competitive delivery market. FoodHub earns a commission on each order: **25%** on orders above \$20, **15%** on orders above \$5, and nothing on orders of \$5 or less.

### Data Dictionary

| Column | Description |
|---|---|
| `order_id` | Unique order identifier |
| `customer_id` | Customer identifier |
| `restaurant_name` | Restaurant name |
| `cuisine_type` | Type of cuisine ordered |
| `cost_of_the_order` | Order cost (\$) |
| `day_of_the_week` | Weekday (Mon–Fri) or Weekend (Sat–Sun) |
| `rating` | Customer rating out of 5, or "Not given" |
| `food_preparation_time` | Restaurant prep time (minutes) |
| `delivery_time` | Time from pickup to drop-off (minutes) |

## 1. Setup and Data Loading

This notebook requires `foodhub_order.csv` at the Google Drive path used below and is configured for Colab. The CSV is not included in this repository. The installation cell pins only part of the environment; SciPy and scikit-learn must also be available (the encoder uses the scikit-learn 1.2+ API). The supplied notebook contains no saved execution outputs. Numerical claims in the narrative should be checked against a complete run on the source data.

In [ ]:
# Pinned versions for reproducibility (Colab)
!pip install numpy==1.25.2 pandas==1.5.3 matplotlib==3.7.1 seaborn==0.13.1 -q --user

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False})
BASE = "#9BB0C8"      # neutral bars
ACCENT = "#1F4E79"    # highlighted bars / key series
pd.set_option("display.max_columns", None)

In [ ]:
from scipy import stats
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import (RepeatedKFold, RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_validate, permutation_test_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)   # single seeded generator -> every bootstrap is reproducible
ALPHA = 0.05                        # two-sided significance level used throughout

scorecard = []                      # each analysis cell logs one row; summarised in section 9
def log_result(question, method, estimate, ci="-", p="-", reading=""):
    scorecard.append({"Question": question, "Method": method, "Estimate": estimate,
                      "95% CI": ci, "p-value": p, "Reading": reading})

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def holm(pvals):
    """Holm-Bonferroni adjusted p-values: controls the family-wise error rate across several tests."""
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty(len(p)); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

def boot_means(x, n_boot=10_000, chunk=1_000):
    """Bootstrap distribution of the sample mean (resampling in chunks to keep memory small)."""
    x = np.asarray(x, float)
    return np.concatenate([rng.choice(x, (chunk, len(x))).mean(axis=1) for _ in range(n_boot // chunk)])

def two_sample_report(a, b, n_boot=10_000):
    """Welch t-test, Mann-Whitney U, Cohen's d and a bootstrap 95% CI for mean(a) - mean(b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    diff = a.mean() - b.mean()
    pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    boot = boot_means(a, n_boot) - boot_means(b, n_boot)
    return {"diff": diff, "ci": tuple(np.percentile(boot, [2.5, 97.5])), "d": diff / pooled, "boot": boot,
            "welch_p": stats.ttest_ind(a, b, equal_var=False).pvalue,
            "mw_p": stats.mannwhitneyu(a, b, alternative="two-sided").pvalue}

def effect_label(d):
    d = abs(d); return "negligible" if d < 0.2 else "small" if d < 0.5 else "medium" if d < 0.8 else "large"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/foodhub_order.csv')
df.head()

## 2. Data Overview and Preparation

In [ ]:
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
print(f"Restaurants: {df['restaurant_name'].nunique()}   Cuisines: {df['cuisine_type'].nunique()}   "
      f"Customers: {df['customer_id'].nunique():,}")
df.dtypes

In [ ]:
df.describe().round(2)

In [ ]:
# The only 'missing' data is ratings recorded as the string 'Not given'
not_rated = (df['rating'] == 'Not given')
print(f"Unrated orders: {not_rated.sum():,} ({not_rated.mean():.1%} of all orders)")
print(f"Any true nulls in the file: {df.isnull().sum().sum()}")

**Preparation choices.** Rather than filling the 39% of missing ratings (mode imputation would distort the rating distribution and could bias associations), I keep them as missing and analyse ratings on rated orders only. I also add two derived columns: a numeric rating and the total time from order placement to delivery.

Section 5.2 compares respondents and non-respondents on observed attributes. Similarity on those attributes cannot establish that missing ratings are ignorable or rule out selection on unobserved satisfaction.

In [ ]:
df['is_rated'] = ~not_rated
df['rating_num'] = pd.to_numeric(df['rating'], errors='coerce')        # 'Not given' -> NaN
df['total_time'] = df['food_preparation_time'] + df['delivery_time']
df[['cost_of_the_order', 'food_preparation_time', 'delivery_time', 'total_time']].describe().round(1)

## 3. Where Is Demand Concentrated?

In [ ]:
cuisine = df['cuisine_type'].value_counts()
share = cuisine / len(df)
top4_share = share.head(4).sum()

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [ACCENT if i < 4 else BASE for i in range(len(cuisine))]
ax.barh(cuisine.index, cuisine.values, color=colors)
ax.invert_yaxis()
for y, (n, p) in enumerate(zip(cuisine.values, share.values)):
    ax.text(n + 5, y, f"{p:.0%}", va='center', fontsize=9)
ax.set_xlim(0, cuisine.max() * 1.1)
ax.set_title(f"Four cuisines account for {top4_share:.0%} of all orders")
ax.set_xlabel("Number of orders"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
top_rest = df['restaurant_name'].value_counts().head(10)
top_rest_share = top_rest / len(df)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [ACCENT] + [BASE] * (len(top_rest) - 1)
ax.barh(top_rest.index, top_rest.values, color=colors)
ax.invert_yaxis()
for y, (n, p) in enumerate(zip(top_rest.values, top_rest_share.values)):
    ax.text(n + 2, y, f"{n} ({p:.1%})", va='center', fontsize=9)
ax.set_xlim(0, top_rest.max() * 1.18)
ax.set_title(f"{top_rest.index[0]} alone receives {top_rest_share.iloc[0]:.1%} of all orders "
             f"(top 10 of {df['restaurant_name'].nunique()} restaurants)")
ax.set_xlabel("Number of orders"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
day_counts = df['day_of_the_week'].value_counts()
weekend_share = day_counts['Weekend'] / len(df)
weekend_top = df.loc[df['day_of_the_week'] == 'Weekend', 'cuisine_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(['Orders'], [day_counts['Weekend']], color=ACCENT, label='Weekend')
ax.barh(['Orders'], [day_counts['Weekday']], left=[day_counts['Weekend']], color=BASE, label='Weekday')
ax.text(day_counts['Weekend'] / 2, 0, f"Weekend\n{weekend_share:.0%}", ha='center', va='center', color='white', fontweight='bold')
ax.text(day_counts['Weekend'] + day_counts['Weekday'] / 2, 0, f"Weekday\n{1 - weekend_share:.0%}", ha='center', va='center')
ax.set_title(f"{weekend_share:.0%} of orders are placed on weekends (2 of 7 days)")
ax.set_xlabel("Number of orders"); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"Most popular weekend cuisine: {weekend_top.index[0]} ({weekend_top.iloc[0]} orders)")

### 3.1 How concentrated is demand, statistically?

Top-N shares are easy to read but depend on where the cutoff is drawn. The Gini coefficient and the effective number of restaurants (the number of equal-sized restaurants that would produce the same concentration) summarise the whole distribution, and an exact binomial test compares weekend volume with an illustrative uniform-day reference. Without dates or sampling coverage, equal exposure across weekdays and weekends is unverified.

In [ ]:
def gini(counts):
    """Gini coefficient of order volume: 0 = every entity equal, 1 = one entity takes everything."""
    x = np.sort(np.asarray(counts, float)); n = len(x)
    return ((2 * np.arange(1, n + 1) - n - 1) @ x) / (n * x.sum())

def effective_n(counts):
    """Inverse Herfindahl index: the number of equal-sized entities that would give the same concentration."""
    s = np.asarray(counts, float) / np.sum(counts); return 1 / (s ** 2).sum()

rest_counts = df['restaurant_name'].value_counts()
cuis_counts = df['cuisine_type'].value_counts()
conc = pd.DataFrame({
    'Entities': [len(rest_counts), len(cuis_counts)],
    'Gini coefficient': [gini(rest_counts), gini(cuis_counts)],
    'Effective number (1/HHI)': [effective_n(rest_counts), effective_n(cuis_counts)],
    'Top-5 share of orders': [rest_counts.head(5).sum() / len(df), cuis_counts.head(5).sum() / len(df)],
}, index=['Restaurants', 'Cuisines']).round(3)
display(conc)

# Lorenz curve for restaurants
sorted_counts = np.sort(rest_counts.to_numpy())
lorenz = np.insert(np.cumsum(sorted_counts) / sorted_counts.sum(), 0, 0)
xs = np.linspace(0, 1, len(lorenz))
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(xs, lorenz, color=ACCENT, lw=2, label='Restaurants')
ax.plot([0, 1], [0, 1], color='grey', ls='--', label='Perfect equality')
ax.fill_between(xs, lorenz, xs, color=BASE, alpha=0.4)
ax.set_title(f"Lorenz curve: Gini = {conc.loc['Restaurants', 'Gini coefficient']:.2f}")
ax.set_xlabel("Cumulative share of restaurants (smallest to largest)"); ax.set_ylabel("Cumulative share of orders")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

# Illustrative reference: uniform order volume across seven days; exposure is not verified.
k, n_all = int((df['day_of_the_week'] == 'Weekend').sum()), len(df)
bt = stats.binomtest(k, n_all, 2 / 7, alternative='greater')
w_ci = stats.binomtest(k, n_all, 2 / 7).proportion_ci(confidence_level=0.95, method='wilson')
print(f"Weekend share {k / n_all:.1%} (95% Wilson CI {w_ci.low:.1%} to {w_ci.high:.1%}) vs {2/7:.1%} expected "
      f"if orders were spread evenly across days; exact binomial p = {fmt_p(bt.pvalue)}")

g, eff = conc.loc['Restaurants', 'Gini coefficient'], conc.loc['Restaurants', 'Effective number (1/HHI)']
log_result("Is restaurant demand concentrated?", "Gini coefficient, effective number (1/HHI)",
           f"Gini {g:.2f}; effective N = {eff:.0f} of {len(rest_counts)}", reading="Volume is unevenly spread across restaurants")
log_result("Is order volume skewed to weekends?", "Exact binomial test vs 2/7, Wilson CI",
           f"{k / n_all:.1%} weekend", f"[{w_ci.low:.1%}, {w_ci.high:.1%}]", fmt_p(bt.pvalue),
           "Weekend share exceeds an even split" if bt.pvalue < ALPHA else "Consistent with an even split")

**Takeaway.** Demand is heavily concentrated: a handful of cuisines and restaurants carry most of the volume, and weekends carry most of the week. This makes targeted restaurant partnerships high-leverage, but it also means losing one top partner would be felt immediately.

## 4. How Fast Do Orders Arrive?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['food_preparation_time'], discrete=True, color=BASE, ax=axes[0])
axes[0].set_title("Prep time is spread evenly from 20 to 35 min")
axes[0].set_xlabel("Food preparation time (minutes)"); axes[0].set_ylabel("Orders")
sns.histplot(df['delivery_time'], discrete=True, color=BASE, ax=axes[1])
axes[1].set_title("Delivery time clusters in two groups")
axes[1].set_xlabel("Delivery time (minutes)"); axes[1].set_ylabel("Orders")
plt.tight_layout(); plt.show()

The two humps in delivery time suggest two different populations of orders. Splitting by day of week explains them:

In [ ]:
means = df.groupby('day_of_the_week')['delivery_time'].agg(['mean', 'std', 'count']).round(2)
gap = means.loc['Weekday', 'mean'] - means.loc['Weekend', 'mean']

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='day_of_the_week', y='delivery_time', order=['Weekday', 'Weekend'],
            hue='day_of_the_week', palette={'Weekday': ACCENT, 'Weekend': BASE},
            legend=False, width=0.5, ax=ax)
for x, day in enumerate(['Weekday', 'Weekend']):
    m = means.loc[day, 'mean']
    ax.scatter(x, m, color='white', edgecolor='black', zorder=3, s=50)
    ax.text(x + 0.3, m, f"mean {m:.1f} min", va='center', fontsize=10)
ax.set_title(f"Weekday deliveries take ~{gap:.0f} minutes longer than weekend deliveries")
ax.set_xlabel(""); ax.set_ylabel("Delivery time (minutes)")
plt.tight_layout(); plt.show()

means.rename(columns={'mean': 'mean_min', 'std': 'std_min', 'count': 'orders'})

### 4.1 Estimate the delivery-time gap and its uncertainty

The boxplot suggests a gap, but a difference in sample means can arise by chance. Three complementary checks are used: the **Welch t-test** (robust to unequal variances), the **Mann-Whitney U test** (no normality assumption), and a **bootstrap confidence interval** on the difference. **Cohen's d** expresses the size of the gap in standard-deviation units. Preparation time is a comparison outcome. A non-significant preparation-time difference does not establish equivalence or distinguish restaurant-side from driver-side causes.

In [ ]:
wd = df[df['day_of_the_week'] == 'Weekday']; we = df[df['day_of_the_week'] == 'Weekend']
rows = {label: two_sample_report(wd[col], we[col]) for label, col in
        [('Delivery time', 'delivery_time'),
         ('Prep time (comparison)', 'food_preparation_time'),
         ('Total time', 'total_time')]}
adj = holm([r['welch_p'] for r in rows.values()])
gap_table = pd.DataFrame({
    'Weekday minus weekend (min)': [r['diff'] for r in rows.values()],
    '95% bootstrap CI': [f"[{r['ci'][0]:.1f}, {r['ci'][1]:.1f}]" for r in rows.values()],
    "Cohen's d": [r['d'] for r in rows.values()],
    'Welch p (Holm-adj.)': [fmt_p(a) for a in adj],
    'Mann-Whitney p': [fmt_p(r['mw_p']) for r in rows.values()],
}, index=rows.keys()).round(2)
display(gap_table)

dr, pr = rows['Delivery time'], rows['Prep time (comparison)']
fig, ax = plt.subplots(figsize=(8, 3.6))
sns.histplot(dr['boot'], bins=50, color=BASE, ax=ax)
ax.axvline(dr['diff'], color=ACCENT, lw=2); ax.axvline(0, color='firebrick', ls='--')
ax.axvspan(*dr['ci'], color=ACCENT, alpha=0.15)
ax.set_title("Bootstrap distribution of the weekday-weekend delivery gap (10,000 resamples)")
ax.set_xlabel("Mean difference in delivery time (minutes); shaded = 95% CI; red = no difference")
plt.tight_layout(); plt.show()

real = dr['welch_p'] < ALPHA and dr['mw_p'] < ALPHA
prep_flat = pr['welch_p'] >= ALPHA
display(Markdown(
    f"**Reading.** Weekday deliveries are **{dr['diff']:.1f} min slower** (95% CI {dr['ci'][0]:.1f} to {dr['ci'][1]:.1f}; "
    f"Cohen's d = {dr['d']:.2f}, a {effect_label(dr['d'])} effect). The gap is "
    f"{'significant under both the Welch t-test and the rank-based Mann-Whitney test, so it does not hinge on normality' if real else 'not significant under both tests, so treat it with caution'}. "
    f"Prep time {'has no statistically detected day-type difference' if prep_flat else 'also differs by day type'} "
    f"(p = {fmt_p(pr['welch_p'])}), which "
    f"{'does not identify the cause of the delivery-time gap' if prep_flat else 'means the restaurant side cannot be ruled out as a contributor'}."))

log_result("Are weekday deliveries slower?", "Welch t-test, Mann-Whitney U, bootstrap CI, Cohen's d",
           f"{dr['diff']:+.1f} min (d = {dr['d']:.2f})", f"[{dr['ci'][0]:.1f}, {dr['ci'][1]:.1f}]", fmt_p(adj[0]),
           "Robust gap" if real else "Not robust")
log_result("Does prep time differ by day type?", "Same battery on prep time",
           f"{pr['diff']:+.1f} min (d = {pr['d']:.2f})", f"[{pr['ci'][0]:.1f}, {pr['ci'][1]:.1f}]", fmt_p(adj[1]),
           "No statistically detected prep-time difference; cause unresolved" if prep_flat else "Prep time differs too")

In [ ]:
over_60 = (df['total_time'] > 60).mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(df['total_time'], discrete=True, color=BASE, ax=ax)
ax.axvline(60, color='firebrick', linestyle='--')
ax.text(60.5, ax.get_ylim()[1] * 0.9, f"{over_60:.1%} of orders\ntake > 60 min", color='firebrick')
ax.set_title(f"Most orders arrive within an hour (mean {df['total_time'].mean():.0f} min from order to door)")
ax.set_xlabel("Total time: preparation + delivery (minutes)"); ax.set_ylabel("Orders")
plt.tight_layout(); plt.show()

**Takeaway.** Nearly 9 in 10 orders arrive within an hour, which supports speed as a selling point. The bigger opportunity is the weekday gap: weekdays have far fewer orders yet slower deliveries, which warrants an operational investigation. Driver supply, delivery distance and routing remain possible explanations; the available data cannot distinguish them.

The gap is assessed statistically in §4.1; the 60-minute share is descriptive, and §8.1 shows how well delivery time can be predicted.

## 5. What Do Ratings Tell Us?

In [ ]:
rated = df[df['is_rated']].copy()
rating_share = rated['rating_num'].value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={'width_ratios': [1, 1.6]})

axes[0].pie([df['is_rated'].sum(), (~df['is_rated']).sum()], labels=['Rated', 'Not rated'],
            colors=[ACCENT, BASE], autopct='%1.0f%%', startangle=90,
            wedgeprops={'width': 0.4}, pctdistance=0.8, textprops={'fontsize': 10})
for t in axes[0].texts[1::2]:
    t.set_color('white'); t.set_fontweight('bold')
axes[0].set_title(f"{(~df['is_rated']).mean():.0%} of orders go unrated")

axes[1].bar(rating_share.index.astype(str), rating_share.values,
            color=[ACCENT if r == 5 else BASE for r in rating_share.index])
for x, p in enumerate(rating_share.values):
    axes[1].text(x, p + 0.01, f"{p:.0%}", ha='center')
axes[1].set_ylim(0, rating_share.max() * 1.15)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Among rated orders, half are 5 stars and none are below 3")
axes[1].set_xlabel("Rating"); axes[1].set_ylabel("Share of rated orders")
plt.tight_layout(); plt.show()

In [ ]:
by_rating = rated.groupby('rating_num')[['delivery_time', 'food_preparation_time', 'total_time', 'cost_of_the_order']].mean().round(1)
by_rating['orders'] = rated['rating_num'].value_counts().sort_index()
display(by_rating)

display(df.groupby('is_rated')[['delivery_time', 'total_time', 'cost_of_the_order']].mean().round(1)
          .rename(index={True: 'Rated', False: 'Not rated'}))

print("Correlation of rating with each variable (rated orders only):")
print(rated[['rating_num', 'delivery_time', 'food_preparation_time', 'total_time', 'cost_of_the_order']]
      .corr()['rating_num'].drop('rating_num').round(3).to_string())

### 5.2 Formal tests: is silence informative, and do ratings track speed or price?

The preceding tables compared averages descriptively. Here the same questions get formal tests. Non-response diagnostics ask whether rated and unrated orders differ on day, cuisine, delivery time, total time or cost (Holm-corrected across the family). **Spearman correlation** is used instead of Pearson because ratings are an ordinal 3-5 scale, and an approximate **power calculation** provides context for sensitivity. The Fisher-z approximation does not account for ordinal ties, clustering or multiple-testing correction; it is not an equivalence test.

In [ ]:
df['cuisine_grp'] = df['cuisine_type'].where(df['cuisine_type'].isin(df['cuisine_type'].value_counts().index[:4]), 'Other')
unr, rtd = df[~df['is_rated']], df[df['is_rated']]

tests = []   # (label, statistic text, raw p)
for col, lab in [('day_of_the_week', 'Rating given vs day type'), ('cuisine_grp', 'Rating given vs cuisine group')]:
    ct = pd.crosstab(df[col], df['is_rated'])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    tests.append((lab + " (chi-square)", f"Cramer's V = {np.sqrt(chi2 / (len(df) * (min(ct.shape) - 1))):.3f}", p))
for col, lab in [('delivery_time', 'Delivery time'), ('total_time', 'Total time'), ('cost_of_the_order', 'Order cost')]:
    r = two_sample_report(rtd[col], unr[col], n_boot=5_000)
    tests.append((f"{lab}: rated vs unrated (Mann-Whitney)", f"diff = {r['diff']:+.2f}, d = {r['d']:.2f}", r['mw_p']))
adj_m = holm([t[2] for t in tests])
miss = pd.DataFrame({'Effect size': [t[1] for t in tests], 'p': [fmt_p(t[2]) for t in tests],
                     'p (Holm-adj.)': [fmt_p(a) for a in adj_m]}, index=[t[0] for t in tests])
display(miss)

# Rank correlations for the observed ordinal ratings.
sp_rows = []
for col in ['delivery_time', 'food_preparation_time', 'total_time', 'cost_of_the_order']:
    rho, p = stats.spearmanr(rated['rating_num'], rated[col]); sp_rows.append((col, rho, p))
sp_adj = holm([r[2] for r in sp_rows])
sp = pd.DataFrame({'Spearman rho': [r[1] for r in sp_rows], 'p': [fmt_p(r[2]) for r in sp_rows],
                   'p (Holm-adj.)': [fmt_p(a) for a in sp_adj]}, index=[r[0] for r in sp_rows]).round(3)
display(sp)

kw = stats.kruskal(*[g['delivery_time'].to_numpy() for _, g in rated.groupby('rating_num')])
n_r = len(rated)
r_min = np.tanh((stats.norm.ppf(0.975) + stats.norm.ppf(0.80)) / np.sqrt(n_r - 3))   # Fisher-z power calculation
print(f"Kruskal-Wallis, delivery time across rating levels: H = {kw.statistic:.2f}, p = {fmt_p(kw.pvalue)}")
print(f"Power: with {n_r:,} rated orders, a correlation of |rho| >= {r_min:.3f} would be detected with 80% power, "
      f"under a Fisher-z approximation; this does not rule out larger population effects.")

n_sig = int((adj_m < ALPHA).sum()); max_rho = max(abs(r[1]) for r in sp_rows)
display(Markdown(
    f"**Reading.** After Holm correction, {n_sig} of {len(tests)} non-response diagnostics are significant, so "
    f"{'non-raters look broadly like raters on the observed variables' if n_sig == 0 else 'non-raters differ from raters on some observed variables, and rating averages should be read with that selection effect in mind'}. "
    f"The strongest rating-vs-operations correlation is |rho| = {max_rho:.2f}, "
    f"{'below the' if max_rho < r_min else 'above the'} {r_min:.2f} detectability threshold."))

log_result("Does non-response signal a bad experience?", "Chi-square, Mann-Whitney, Holm correction",
           f"{n_sig} of {len(tests)} tests significant", p=fmt_p(adj_m.min()),
           reading="No evidence of selective non-response on observed variables" if n_sig == 0 else "Some selection on observed variables")
log_result("Do ratings track speed or price?", "Spearman rho, Holm correction, power analysis",
           f"max |rho| = {max_rho:.2f}", p=fmt_p(sp_adj.min()),
           reading=f"Approximate 80%-power threshold: |rho| {r_min:.2f}; not an exclusion bound")

**Takeaway.** Customers who leave ratings are overwhelmingly satisfied, and unrated orders look the same as rated ones: both average about 24 minutes for delivery and \$16–17 per order. The observed comparisons cannot determine how satisfied non-respondents were.

Ratings also barely move with speed or price. 3-, 4- and 5-star orders all average about 24 minutes for delivery and 27 minutes for prep, and every correlation is close to zero. Food quality, order accuracy and packaging are candidate variables for future collection. Weak observed correlations do not establish that all relevant relationships are absent.

Section 5.2 reports the tests, effect sizes and approximate power calculation supporting this assessment.

## 6. Where Does Revenue Come From?

In [ ]:
cost = df['cost_of_the_order']
# Tiered commission: 25% on orders > $20, 15% on orders > $5, 0% otherwise
df['revenue'] = np.select([cost > 20, cost > 5], [cost * 0.25, cost * 0.15], default=0)
df['tier'] = np.select([cost > 20, cost > 5], ['> $20 (25%)', '$5–20 (15%)'], default='≤ $5 (0%)')

total_revenue = df['revenue'].sum()
tier_summary = df.groupby('tier').agg(orders=('revenue', 'size'), revenue=('revenue', 'sum'))
tier_summary['share_of_orders'] = (tier_summary['orders'] / len(df)).round(3)
tier_summary['share_of_revenue'] = (tier_summary['revenue'] / total_revenue).round(3)

print(f"Total FoodHub revenue across {len(df):,} orders: ${total_revenue:,.2f}")
print(f"Average revenue per order: ${total_revenue / len(df):.2f}")
tier_summary.round(2)

In [ ]:
tiers = ['$5–20 (15%)', '> $20 (25%)']
ts = tier_summary.reindex(tiers).fillna(0)

fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(2)
ax.barh(x + 0.2, ts['share_of_orders'], height=0.4, color=BASE, label='Share of orders')
ax.barh(x - 0.2, ts['share_of_revenue'], height=0.4, color=ACCENT, label='Share of revenue')
for i, t in enumerate(tiers):
    ax.text(ts.loc[t, 'share_of_orders'] + 0.01, i + 0.2, f"{ts.loc[t, 'share_of_orders']:.0%}", va='center')
    ax.text(ts.loc[t, 'share_of_revenue'] + 0.01, i - 0.2, f"{ts.loc[t, 'share_of_revenue']:.0%}", va='center')
ax.set_yticks(x); ax.set_yticklabels(tiers)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlim(0, 1)
hi = ts.loc['> $20 (25%)']
ax.set_title(f"Orders over $20 are {hi['share_of_orders']:.0%} of volume but {hi['share_of_revenue']:.0%} of revenue")
ax.legend(loc='lower right', frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
order = df.groupby('cuisine_type')['cost_of_the_order'].median().sort_values(ascending=False).index
counts = df['cuisine_type'].value_counts()

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=df, y='cuisine_type', x='cost_of_the_order', order=order, color=BASE, ax=ax)
ax.axvline(20, color='firebrick', linestyle='--')
ax.set_yticks(range(len(order)), [f"{c} (n={counts[c]})" for c in order])
ax.set_title("Pricier cuisines (e.g. French, Thai) are among the least ordered")
ax.set_xlabel("Order cost ($)   |   red dashed line = $20 commission threshold"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

### 6.1 How certain is the revenue concentration, and does price relate to popularity?

A single sample yields a single percentage. Bootstrapping over orders gives a confidence interval around the revenue shares, and rank-based tests check whether cuisines genuinely differ in price and whether pricier cuisines are less popular.

In [ ]:
rev = df['revenue'].to_numpy(); hi_mask = (df['cost_of_the_order'] > 20).to_numpy(); n = len(df)
B, chunk = 5_000, 500
sh_rev, sh_ord, tot = [], [], []
for _ in range(B // chunk):
    idx = rng.integers(0, n, (chunk, n)); r, h = rev[idx], hi_mask[idx]
    sh_rev.append((r * h).sum(1) / r.sum(1)); sh_ord.append(h.mean(1)); tot.append(r.sum(1))
sh_rev, sh_ord, tot = map(np.concatenate, (sh_rev, sh_ord, tot))
ci = lambda a: np.percentile(a, [2.5, 97.5])
boot_tbl = pd.DataFrame({
    'Point estimate': [hi_mask.mean(), rev[hi_mask].sum() / rev.sum(), rev.sum()],
    '95% CI low': [ci(sh_ord)[0], ci(sh_rev)[0], ci(tot)[0]],
    '95% CI high': [ci(sh_ord)[1], ci(sh_rev)[1], ci(tot)[1]]},
    index=['Share of orders above $20', 'Share of revenue from orders above $20', 'Total revenue ($)']).round(3)
display(boot_tbl)

groups = [g['cost_of_the_order'].to_numpy() for _, g in df.groupby('cuisine_type')]
H, p_kw = stats.kruskal(*groups)
eps2 = (H - len(groups) + 1) / (len(df) - len(groups))          # epsilon-squared effect size
cu = df.groupby('cuisine_type').agg(orders=('order_id', 'size'), median_cost=('cost_of_the_order', 'median'))
rho_c, p_c = stats.spearmanr(cu['orders'], cu['median_cost'])
print(f"Kruskal-Wallis, order cost across {len(groups)} cuisines: H = {H:.1f}, p = {fmt_p(p_kw)}, epsilon^2 = {eps2:.3f}")
print(f"Cuisine popularity vs median price (n = {len(cu)} cuisines): Spearman rho = {rho_c:.2f}, p = {fmt_p(p_c)}")

log_result("How much revenue comes from orders above $20?", "Bootstrap over orders (5,000 resamples)",
           f"{rev[hi_mask].sum() / rev.sum():.0%} of revenue from {hi_mask.mean():.0%} of orders",
           f"[{ci(sh_rev)[0]:.0%}, {ci(sh_rev)[1]:.0%}]", reading="Revenue concentration is statistically stable")
log_result("Are pricier cuisines less popular?", "Spearman across cuisines; Kruskal-Wallis on cost",
           f"rho = {rho_c:.2f} (n = {len(cu)})", p=fmt_p(p_c),
           reading="Supported" if p_c < ALPHA and rho_c < 0 else "Not statistically supported with so few cuisines")

**Takeaway.** The typical order sits below the \$20 threshold, yet orders above it produce a disproportionate share of revenue. Small nudges that lift baskets over \$20 (add-on suggestions, bundle deals, free-delivery minimums) could shift orders into the higher commission tier under the stated rules. Test their effects on conversion, discounts and contribution margin before rollout.

## 7. Promotion Eligibility

FoodHub wants to promote restaurants with **more than 50 ratings** and an **average rating above 4**.

In [ ]:
rest_ratings = rated.groupby('restaurant_name')['rating_num'].agg(rating_count='count', avg_rating='mean')
eligible = (rest_ratings[(rest_ratings['rating_count'] > 50) & (rest_ratings['avg_rating'] > 4)]
            .sort_values('avg_rating', ascending=False).round(2))
print(f"{len(eligible)} restaurants qualify for the promotion:")
eligible

### 7.1 Are the promotion picks robust to sampling noise?

The eligibility rule uses a raw average, but an average over 51 ratings is much less certain than one over 150. For each eligible restaurant, a one-sided t-test (Holm-corrected) checks whether its true mean rating plausibly exceeds 4.0, and a watch list flags near-eligible restaurants.

In [ ]:
cand = rest_ratings[rest_ratings['rating_count'] > 50].index
rows7 = []
for name in cand:
    r = rated.loc[rated['restaurant_name'] == name, 'rating_num'].to_numpy()
    half = stats.t.ppf(0.975, len(r) - 1) * stats.sem(r)
    rows7.append({'restaurant': name, 'ratings': len(r), 'avg': r.mean(), 'CI low': r.mean() - half,
                  'CI high': r.mean() + half, 'p (mean > 4)': stats.ttest_1samp(r, 4, alternative='greater').pvalue})
cand_tbl = pd.DataFrame(rows7).set_index('restaurant')
if len(cand_tbl):
    cand_tbl['p (Holm-adj.)'] = holm(cand_tbl['p (mean > 4)'])
    cand_tbl['Clears 4.0 with confidence'] = cand_tbl['p (Holm-adj.)'] < ALPHA
    display(cand_tbl.round(3))
n_robust = int(cand_tbl['Clears 4.0 with confidence'].sum()) if len(cand_tbl) else 0

watch = rest_ratings[(rest_ratings['rating_count'].between(30, 50)) & (rest_ratings['avg_rating'] > 4)].round(2)
print(f"Watch list: {len(watch)} restaurants have an average above 4 but 30-50 ratings (below the 50-rating bar):")
display(watch)

log_result("Are promotion picks robustly above 4.0?", "One-sided t-test vs 4.0, Holm correction, t-interval",
           f"{n_robust} of {len(cand_tbl)} eligible restaurants", p="-",
           reading="Picks are statistically defensible" if n_robust == len(cand_tbl) else "Some picks are within sampling noise of 4.0")

## 8. Predictive Modeling: What Can the Data Explain?

Three modeling questions, each evaluated on data the model never saw during training:

1. **Can delivery time be predicted** from order attributes, and what drives it? (§8.1)
2. **Does the weekend effect survive adjustment** for cuisine, order size and prep time? (§8.2)
3. **Can anything in the data predict a 5-star rating?** (§8.3)

**Method notes.** All preprocessing (one-hot encoding, scaling) sits inside scikit-learn pipelines, so validation rows do not fit their own preprocessing. Random folds can still share customers and restaurants; grouped validation is needed to assess transfer to unseen entities. Every model is compared against a naive baseline, because a model is only useful if it beats guessing the average. Because the data have no dates, time-based validation is not possible; random folds are used.

### 8.1 Predicting delivery time

Five model specifications of differing flexibility (day type only, ridge regression, random forest, gradient boosting) compete under repeated 5-fold cross-validation. Delivery time is measured from pickup, so observed preparation time is an input for a prediction made at pickup, not at initial order placement; total time is excluded because it contains the target.

In [ ]:
num_feats = ['cost_of_the_order', 'food_preparation_time']
cat_feats = ['day_of_the_week', 'cuisine_type', 'restaurant_name']
X, y = df[num_feats + cat_feats], df['delivery_time']

def preprocessor(num, cat, scale=False):
    """Encoding happens inside each CV fold; rare restaurants (<20 orders) are pooled automatically."""
    parts = []
    if num: parts.append(('num', StandardScaler() if scale else 'passthrough', num))
    if cat: parts.append(('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20,
                                               sparse_output=False), cat))
    return ColumnTransformer(parts)

models = {
    'Baseline (predict the mean)': Pipeline([('pre', preprocessor(num_feats, cat_feats)), ('m', DummyRegressor())]),
    'Day type only (linear)': Pipeline([('pre', preprocessor([], ['day_of_the_week'])), ('m', LinearRegression())]),
    'Ridge regression': Pipeline([('pre', preprocessor(num_feats, cat_feats, scale=True)), ('m', Ridge(alpha=10))]),
    'Random forest': Pipeline([('pre', preprocessor(num_feats, cat_feats)),
                               ('m', RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=SEED, n_jobs=-1))]),
    'Gradient boosting': Pipeline([('pre', preprocessor(num_feats, cat_feats)),
                                   ('m', GradientBoostingRegressor(n_estimators=300, learning_rate=0.03, max_depth=3,
                                                                   subsample=0.8, random_state=SEED))]),
}
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)
scoring = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}
cv_rows = {}
for name, model in models.items():
    s = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows[name] = {'RMSE (min)': -s['test_rmse'].mean(), 'RMSE sd': s['test_rmse'].std(),
                     'MAE (min)': -s['test_mae'].mean(), 'R2': s['test_r2'].mean()}
cv_tbl = pd.DataFrame(cv_rows).T.sort_values('RMSE (min)')
base_rmse = cv_tbl.loc['Baseline (predict the mean)', 'RMSE (min)']
cv_tbl['RMSE gain vs baseline'] = 1 - cv_tbl['RMSE (min)'] / base_rmse
display(cv_tbl.round(3))

fig, ax = plt.subplots(figsize=(9, 3.8))
colors = [ACCENT if i == 0 else BASE for i in range(len(cv_tbl))]
ax.barh(cv_tbl.index, cv_tbl['RMSE (min)'], xerr=cv_tbl['RMSE sd'], color=colors, capsize=3)
ax.axvline(base_rmse, color='firebrick', ls='--'); ax.invert_yaxis()
ax.set_title("Cross-validated error in predicting delivery time (lower is better)")
ax.set_xlabel("RMSE in minutes (15 folds; bars = 1 sd; dashed = naive baseline)")
plt.tight_layout(); plt.show()

# Exploratory importance on a separate fit; this split was included in the earlier model comparison.
best_name = cv_tbl.index[0]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED)
best = clone(models[best_name]).fit(X_tr, y_tr)
imp = permutation_importance(best, X_te, y_te, scoring='neg_root_mean_squared_error', n_repeats=30,
                             random_state=SEED, n_jobs=-1)
imp_tbl = pd.DataFrame({'increase_in_RMSE': imp.importances_mean, 'sd': imp.importances_std},
                       index=X.columns).sort_values('increase_in_RMSE', ascending=False)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.barh(imp_tbl.index, imp_tbl['increase_in_RMSE'], xerr=imp_tbl['sd'], color=ACCENT, capsize=3); ax.invert_yaxis()
ax.set_title(f"Permutation importance ({best_name}, held-out data)")
ax.set_xlabel("Increase in RMSE (minutes) when the feature is shuffled")
plt.tight_layout(); plt.show()

top_feat = imp_tbl.index[0]
display(Markdown(
    f"**Reading.** The best model ({best_name}) cuts prediction error by **{cv_tbl.iloc[0]['RMSE gain vs baseline']:.0%}** versus "
    f"guessing the average (CV R-squared = {cv_tbl.iloc[0]['R2']:.2f}). The most influential input is **{top_feat}**. "
    f"{'Day type alone captures most of the achievable accuracy, so the operational story is simple.' if cv_tbl.loc['Day type only (linear)', 'RMSE (min)'] <= cv_tbl['RMSE (min)'].min() * 1.03 else 'Richer features add accuracy beyond day type alone.'}"))
log_result("Can delivery time be predicted from order attributes?", "Ridge / random forest / gradient boosting, repeated 5-fold CV",
           f"{best_name}: RMSE {cv_tbl.iloc[0]['RMSE (min)']:.1f} min, R2 {cv_tbl.iloc[0]['R2']:.2f}",
           reading=f"{cv_tbl.iloc[0]['RMSE gain vs baseline']:.0%} lower error than baseline; top driver: {top_feat}")

### 8.2 Is the weekend effect explained by something else?

Weekdays might be slower simply because they have a different cuisine mix, larger orders or longer prep times. A linear regression that holds those factors constant estimates the adjusted association with weekend status, not a causal effect, with a bootstrap confidence interval.

In [ ]:
Xd = pd.get_dummies(df[['day_of_the_week', 'cuisine_type']], drop_first=True).astype(float)
Xd['cost_of_the_order'] = df['cost_of_the_order'].to_numpy(); Xd['food_preparation_time'] = df['food_preparation_time'].to_numpy()
Xd.insert(0, 'const', 1.0)
A, yv = Xd.to_numpy(), df['delivery_time'].to_numpy(float); j = list(Xd.columns).index('day_of_the_week_Weekend')
beta = np.linalg.lstsq(A, yv, rcond=None)[0]
boots = np.empty(2_000)
for b in range(len(boots)):
    i = rng.integers(0, len(yv), len(yv)); boots[b] = np.linalg.lstsq(A[i], yv[i], rcond=None)[0][j]
adj_lo, adj_hi = np.percentile(boots, [2.5, 97.5])
raw = -rows['Delivery time']['diff']   # weekend minus weekday, from section 4.1
print(f"Raw weekend effect: {raw:+.2f} min   |   Adjusted for cuisine, cost and prep time: {beta[j]:+.2f} min "
      f"(95% bootstrap CI {adj_lo:+.2f} to {adj_hi:+.2f})")
log_result("Does the weekend effect survive adjustment?", "OLS with cuisine, cost, prep time; bootstrap CI (2,000)",
           f"{beta[j]:+.1f} min (raw {raw:+.1f})", f"[{adj_lo:+.1f}, {adj_hi:+.1f}]",
           reading="Effect persists after adjustment" if adj_hi < 0 or adj_lo > 0 else "Effect not distinguishable from zero after adjustment")

### 8.3 Can order data predict satisfaction?

Section 5 found that ratings barely correlate with speed or price. A classifier is a stronger test, since it can combine every variable at once and find non-linear patterns. If these models perform near chance, this workflow has found limited predictive signal; that does not prove the available features contain none. The target is a five-star rating among respondents, not satisfaction across all orders. A **permutation test** (shuffling the labels 200 times) provides a p-value for "better than chance".

In [ ]:
rm = df[df['is_rated']].copy(); rm['five_star'] = (rm['rating_num'] == 5).astype(int)
cf_num = ['cost_of_the_order', 'food_preparation_time', 'delivery_time']
cf_cat = ['day_of_the_week', 'cuisine_type', 'restaurant_name']
Xc, yc = rm[cf_num + cf_cat], rm['five_star']

clfs = {
    'Baseline (class prior)': Pipeline([('pre', preprocessor(cf_num, cf_cat)), ('m', DummyClassifier(strategy='prior'))]),
    'Logistic regression': Pipeline([('pre', preprocessor(cf_num, cf_cat, scale=True)),
                                     ('m', LogisticRegression(C=0.5, max_iter=2000))]),
    'Gradient boosting': Pipeline([('pre', preprocessor(cf_num, cf_cat)),
                                   ('m', GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=2,
                                                                    subsample=0.8, random_state=SEED))]),
}
cv_c = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
auc = {}
for name, m in clfs.items():
    s = cross_validate(m, Xc, yc, cv=cv_c, scoring='roc_auc', n_jobs=-1)['test_score']
    auc[name] = {'ROC AUC': s.mean(), 'sd': s.std()}
auc_tbl = pd.DataFrame(auc).T; display(auc_tbl.round(3))

score, _, p_perm = permutation_test_score(clfs['Logistic regression'], Xc, yc, scoring='roc_auc',
                                          cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                                          n_permutations=200, random_state=SEED, n_jobs=-1)
best_auc = auc_tbl['ROC AUC'].max()
print(f"Permutation test for logistic regression (200 label shuffles): AUC = {score:.3f}, p = {fmt_p(p_perm)}")
display(Markdown(
    f"**Reading.** Best cross-validated AUC is **{best_auc:.2f}** (0.50 = coin flip, 1.00 = perfect). "
    f"{'Operational and menu data carry little signal about which orders earn five stars, under this validation design; it does not establish that satisfaction drivers are absent.' if best_auc < 0.65 else 'There is usable signal, so a richer rating model may be worth building.'}"))
log_result("Can order data predict a 5-star rating?", "Logistic regression and gradient boosting, repeated CV; permutation test",
           f"best AUC = {best_auc:.2f}", p=fmt_p(p_perm),
           reading="Limited predictive signal in the evaluated models" if best_auc < 0.65 else "Meaningful signal")

**Takeaway.** Model results above quantify how much of delivery-time variation order attributes explain and which inputs matter most, and the rating classifier shows whether satisfaction can be anticipated from operational data. Predictive accuracy here is about *explaining* patterns in a historical snapshot, not about forecasting future demand; with no dates in the data, no such forecast can be validated.

## 9. Evidence Scorecard

Every test and model from sections 3-8, consolidated. The table is built from the results computed above, so it always matches the data.

In [ ]:
sc = pd.DataFrame(scorecard)
pd.set_option('display.max_colwidth', None)
display(sc)

## 10. Quick Answers to Key Business Questions

In [ ]:
top_customers = df['customer_id'].value_counts().head(3)
answers = pd.Series({
    "Top 5 restaurants by orders": ", ".join(df['restaurant_name'].value_counts().head(5).index),
    "Most popular weekend cuisine": f"{weekend_top.index[0]} ({weekend_top.iloc[0]} orders)",
    "Orders costing more than $20": f"{(cost > 20).mean():.1%}",
    "Mean delivery time": f"{df['delivery_time'].mean():.1f} min",
    "Mean total time (prep + delivery)": f"{df['total_time'].mean():.1f} min",
    "Orders taking over 60 min in total": f"{over_60:.1%}",
    "Top 3 customers (orders)": ", ".join(f"{c} ({n})" for c, n in top_customers.items()),
    "Top 3 customers' share of orders": f"{top_customers.sum() / len(df):.1%}",
    "Total FoodHub revenue": f"${total_revenue:,.2f}",
    "Restaurants eligible for promotion": ", ".join(eligible.index),
}, name="Answer")
pd.set_option("display.max_colwidth", None)
answers.to_frame()

## 11. Conclusions and Recommendations

**1. Protect and deepen top restaurant partnerships.** Four cuisines drive 83% of orders and the top five restaurants alone account for about a third (see the Gini and effective-number measures in §3.1). Prioritise these partners (featured placement, co-marketing, the promotion above), while recruiting more restaurants in under-served cuisines to reduce dependency on a few names.

**2. Close the weekday delivery gap.** Weekday deliveries average about 6 minutes longer than weekend deliveries even though weekdays have far less volume. §4.1 tests the gap with parametric, rank-based and bootstrap methods, compares prep times, and §8.2 checks that it survives adjustment for cuisine, order size and prep time. Investigate driver availability, distance and routing on weekdays. If staffing is implicated, test a driver-incentive pilot with a control group before broader rollout.

**3. Lift more orders over \$20.** Orders above \$20 are 29% of volume but 60% of revenue, and §6.1 puts a bootstrap confidence interval around that share. Bundle suggestions, add-on prompts at checkout, or a free-delivery threshold just above \$20 could move orders into the higher commission tier; assess incremental margin and conversion in a controlled pilot.

**4. Rebuild the feedback loop.** 39% of orders go unrated and the ratings that exist are clustered at 4-5 stars. §5.2 tests whether silence is informative, and §8.3 shows how well any available variable predicts a 5-star rating. Weak predictability motivates additional measurement but does not prove that satisfaction drivers are absent from the current features. A one-tap post-delivery prompt, plus a short optional question on what went well or badly, would give FoodHub data it can act on.

**5. Market on speed and choice.** With nearly 90% of orders arriving within an hour and 178 restaurants across 14 cuisines, speed and variety are candidate value propositions. Verify that the sample represents current operations before making service promises.

**6. Treat promotion picks with statistical discipline.** §7.1 shows which eligible restaurants clear the 4.0 bar with confidence and which sit within sampling noise of it. Re-run the eligibility check periodically, and consider a watch list for restaurants approaching 50 ratings.

### Limitations and Next Steps

- **Observational snapshot.** The data cover a single period with no dates, so trends and seasonality cannot be assessed, time-based validation is impossible, and all findings are associations rather than proven causes.
- **Missing drivers.** There is no information on delivery distance, driver counts, or customer acquisition cost, so the mechanism behind the weekday gap and true profitability remain hypotheses. Preparation-time comparisons and covariate adjustment do not identify the cause.
- **Ratings.** They span only 3-5, are missing for 39% of orders, and are ordinal; non-response diagnostics can only examine observed variables, so selection on unobserved factors cannot be ruled out.
- **Validation and dependence.** Repeated customers and restaurants may induce dependence. The tests and order-level bootstraps do not adjust for clustering, and random CV does not establish performance on new customers or restaurants. The permutation-importance split was drawn after model comparison on all rows, so it is exploratory rather than an untouched final test.
- **Multiple testing.** Holm correction is applied within each family of tests, but the report as a whole contains many tests; p-values near 0.05 should be treated as suggestive.
- **Next steps.** (1) Add timestamps, pickup/drop-off coordinates and driver availability to enable a causal design (for example, a weekday driver-incentive pilot with a control group). (2) Add a richer feedback instrument and re-run §8.3. (3) With dated data, build demand forecasts validated with time-based splits.